<a href="https://colab.research.google.com/github/Waleedamin008/Pet-medicine-prediction/blob/main/Ai_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_excel("Data set for project ai.xlsx")

print(df.head())
print(df.info())
print(df.describe())

In [ ]:
print(df.isnull().sum())

In [ ]:
sns.countplot(x='Pet_Type', data=df)
plt.title("Pet Type Distribution")
plt.show()

In [ ]:
sns.countplot(y='Disease_Reported_By_Customer', data=df)
plt.title("Disease Distribution")
plt.show()

In [ ]:
sns.histplot(df['Pet_Age_Years'], kde=True)
plt.title("Pet Age Distribution")
plt.show()

In [ ]:
X = df[['Pet_Type', 'Pet_Age_Years', 'Disease_Reported_By_Customer']]
y = df['Target_Medicine']

In [ ]:

sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
le_pet = LabelEncoder()
le_disease = LabelEncoder()
le_target = LabelEncoder()

X['Pet_Type'] = le_pet.fit_transform(X['Pet_Type'])
X['Disease_Reported_By_Customer'] = le_disease.fit_transform(X['Disease_Reported_By_Customer'])

y = le_target.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
svm_model = SVC(probability=True)
svm_model.fit(X_train, y_train)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

In [ ]:
models = {
    "SVM": svm_model,
    "KNN": knn_model,
    "Random Forest": rf_model
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(name, "Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(name + " Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    print("\n", name, "Classification Report\n")
    print(classification_report(y_test, y_pred))

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    print("\n", name, "Classification Report\n")
    print(classification_report(y_test, y_pred))

In [ ]:
y_test_bin = label_binarize(y_test, classes=np.unique(y))
n_classes = y_test_bin.shape[1]

for name, model in models.items():
    y_score = model.predict_proba(X_test)

    plt.figure()

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        plt.plot(fpr, tpr, label='Class ' + str(i))

    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(name + " ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append([name, acc])

results_df = pd.DataFrame(results, columns=["Model", "Accuracy"])
print(results_df)